In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df = pd.DataFrame({
    "Mathematics": [12, 15, 8, 17, 10, 14],
    "Statistics":  [11, 14, 9, 16, 12, 15],
    "Programming": [14, 16, 10, 18, 11, 13],
    "Algorithms":  [13, 15, 9, 17, 10, 14],
}, index=["S1", "S2", "S3", "S4", "S5", "S6"])
df

,Mathematics,Statistics,Programming,Algorithms
S1,12,11,14,13
S2,15,14,16,15
S3,8,9,10,9
S4,17,16,18,17
S5,10,12,11,10
S6,14,15,13,14


In [ ]:
print("Shape:", df.shape)
df.describe()

In [ ]:
X = df.values.astype(float)
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
pd.DataFrame(Z, index=df.index, columns=df.columns).round(3)

In [ ]:
R = np.corrcoef(Z, rowvar=False)
pd.DataFrame(R, index=df.columns, columns=df.columns).round(3)

In [ ]:
R = np.corrcoef(Z, rowvar=False)
pd.DataFrame(R, index=df.columns, columns=df.columns).round(3)

In [ ]:
eigvals, eigvecs = np.linalg.eigh(R)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
W = eigvecs[:, :2]
print("Eigenvalues:", eigvals.round(4))
print("Selected eigenvectors:\n", W.round(4))

In [ ]:
scores_scratch = Z @ W
print(pd.DataFrame(scores_scratch, index=df.index, columns=["PC1", "PC2"]).round(4))

evr_scratch = eigvals / eigvals.sum()
cum_scratch = np.cumsum(evr_scratch)
print("Explained variance ratio:", evr_scratch.round(4))
print("Cumulative:", cum_scratch.round(4))

In [ ]:
Z_sk = StandardScaler().fit_transform(df)
pca = PCA(n_components=2)
scores_sk = pca.fit_transform(Z_sk)

print(pd.DataFrame(scores_sk, index=df.index, columns=["PC1", "PC2"]).round(4))
print("Explained variance ratio:", pca.explained_variance_ratio_.round(4))
print("Cumulative:", pca.explained_variance_ratio_.cumsum().round(4))
print("Eigenvalues (explained_variance_):", pca.explained_variance_.round(4))

In [ ]:
print("Same coordinates up to sign:", np.allclose(np.abs(scores_scratch), np.abs(scores_sk)))
for k in range(2):
    s = np.sign(np.dot(scores_scratch[:, k], scores_sk[:, k]))
    print(f"PC{k+1}: sign factor scratch vs sklearn = {int(s)}")

print("\nEigenvalues scratch:", eigvals[:2].round(4))
print("Eigenvalues sklearn:", pca.explained_variance_.round(4))
print("Ratio sklearn/scratch:", (pca.explained_variance_ / eigvals[:2]).round(4))
print("\nEVR scratch:", evr_scratch[:2].round(4))
print("EVR sklearn:", pca.explained_variance_ratio_.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, S, title in zip(axes, [scores_scratch, scores_sk], ["From scratch", "scikit-learn"]):
    ax.scatter(S[:, 0], S[:, 1])
    for i, name in enumerate(df.index):
        ax.annotate(name, (S[i, 0], S[i, 1]), textcoords="offset points", xytext=(5, 5))
    ax.axhline(0, lw=0.5, color="gray")
    ax.axvline(0, lw=0.5, color="gray")
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("pca_comparison.png", dpi=150)
plt.show()

In [ ]:
print(f"Variance retained by PC1 + PC2: {cum_scratch[1] * 100:.2f}%")

In [ ]:
print(f"Variance retained by PC1 + PC2: {cum_scratch[1] * 100:.2f}%")